# 02 Zero shot fiction classification

**Question:** Can a pretrained model assign a broad category without training on this book dataset? We use natural language inference to compare candidate label hypotheses.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "book_recommender").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from book_recommender.config import RAW, CLEAN, CLASSIFIED, ENRICHED, REPORTS


In [2]:
books = pd.read_csv(CLEAN, dtype={"isbn13": str, "book_id": str})
books[["title", "categories"]].head()

,title,categories
0,Gilead,Fiction
1,Spider's Web,Detective and mystery stories
2,Rage of angels,Fiction
3,The Four Loves,Christian life
4,The Problem of Pain,Christian life


## From labels to hypotheses
The premise is the title and description. The candidates become `This book is Fiction.` and `This book is Non-fiction.` The NLI model compares entailment evidence. Zero-shot means no task-specific training on these books; the pretrained model has seen other labelled NLI data.

In [3]:
from book_recommender.text_classification import classify_books
example = classify_books(books.head(8), batch_size=4)
example[["title", "categories", "predicted_category", "category_confidence", "category_needs_review"]]

classification: 0it [00:00, ?it/s]

classification: 0it [00:00, ?it/s]

,title,categories,predicted_category,category_confidence,category_needs_review
0,Gilead,Fiction,Fiction,0.999060,False
1,Spider's Web,Detective and mystery stories,Fiction,0.995649,False
2,Rage of angels,Fiction,Fiction,0.984197,False
3,The Four Loves,Christian life,Non-fiction,0.538101,True
4,The Problem of Pain,Christian life,Non-fiction,0.946434,False
5,Empires of the Monsoon,"Africa, East",Non-fiction,0.823538,False
6,The Gap Into Madness,"Hyland, Morn (Fictitious character)",Fiction,0.987444,False
7,Master of the Game,Adventure stories,Fiction,0.955593,False


## Review uncertainty
The winning score is relative to these two labels. It is not calibrated confidence in factual correctness. Biographies written as narratives, poetry, and mixed collections may be difficult. We flag scores below 0.65 for manual review.

In [4]:
example.sort_values("category_confidence")[["title", "description", "predicted_category", "category_confidence"]].head(3)

,title,description,predicted_category,category_confidence
3,The Four Loves,Lewis' work on the nature of love divides love...,Non-fiction,0.538101
5,Empires of the Monsoon,Until Vasco da Gama discovered the sea-route t...,Non-fiction,0.823538
4,The Problem of Pain,"""In The Problem of Pain, C.S. Lewis, one of th...",Non-fiction,0.946434


## Build the catalog
For all books run `python -m scripts.build_pipeline --stage classify` in the root. Use `--limit 200` for a first run. Predictions are cached by text/model/settings; rerunning a completed batch does not repeat inference. The demonstration above deliberately does not overwrite the full catalog file.

In [5]:
if CLASSIFIED.exists():
    catalog = pd.read_csv(CLASSIFIED)
    display(catalog.predicted_category.value_counts().to_frame("books"))
else:
    print("Run the classify stage to create the catalog.")

,books
predicted_category,
Fiction,3788
Non-fiction,1941


## Evaluation exercise
Manually label a random set of books, blind to predictions. Calculate precision/recall for both categories and inspect a confusion matrix. Avoid treating imperfect source genres as unquestioned ground truth.